# Notebook 02 — Multi-seed factorial attention ablation and bootstrap analysis

Primary experiment of the manuscript: a fully crossed 4 x 3 factorial design
on one fixed YOLOv12s-cls backbone.

- Variants: no attention (baseline), squeeze-and-excitation, CBAM, LTA
- Seeds: 42, 123, 2026
- 12 training runs, identical in data, schedule, augmentation,
  checkpoint-selection rule and evaluation code

Prerequisite: run steps 1, 2 and 5 of Notebook 01 first, so that Drive is
mounted, the dataset is unpacked at `/content/dataset`, and Ultralytics is
installed. Then run the cells below in order.

Environment recorded by the first cell into `environment_and_protocol.json`:
Python 3.13.15, Ultralytics 8.4.131, PyTorch 2.11.0+cu128, single Tesla T4.
This differs from Notebook 01 because Colab's default image had advanced
between the two campaigns; both environments are reported in the manuscript.

What this notebook produces
- `environment_and_protocol.json` — versions, seeds, split counts, pretrained
  checkpoint MD5, and every non-39-class item quarantined out of the dataset
- 12 run directories, each written only after a verified completion marker
  (incomplete runs are renamed `*_INVALID_OR_INCOMPLETE_*` and excluded)
- `predictions/*.csv` — per-image prediction records for all 636 test images,
  one file per run, plus one for EfficientNet-B0
- `tables/*.csv` — per-run metrics, multi-seed summaries, bootstrap accuracy
  intervals and paired within-seed difference tests (Tables 3, 4, 5)
- `result_manifest.json` — size and MD5 of every released artefact

Outcome: no variant's paired within-seed difference against the baseline
excludes zero. The +1.42 pp single-seed LTA gain measured in Notebook 01 does
not reproduce across seeds.


In [ ]:
# ============================================================
# Step 1: environment check, dataset gating and shared configuration
# ============================================================

import os
import sys
import json
import time
import random
import hashlib
import platform
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from ultralytics import YOLO

DATA_ROOT = "/content/dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "val")
TEST_DIR = os.path.join(DATA_ROOT, "test")

PRETRAINED_WEIGHT = "/content/yolov12s-cls.pt"
PRETRAINED_URL = "https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12s-cls.pt"

SUPP_ROOT = "/content/drive/MyDrive/TCM_YOLOv12_runs/supplementary_attention_ablation"
PRED_DIR = os.path.join(SUPP_ROOT, "predictions")
TABLE_DIR = os.path.join(SUPP_ROOT, "tables")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15

# Full experiment: 4 variants x 3 seeds = 12 training runs.
# SEEDS can be temporarily reduced to [42] for a smoke test, but the
# reported results require all three seeds below to be run to completion.
SEEDS = [42, 123, 2026]
VARIANTS = ["baseline", "SE", "CBAM", "LTA"]

for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not os.path.isdir(p):
        raise FileNotFoundError(
            f"{p} not found. Run steps 1-2 of notebook 01 first so that the "
            f"dataset is unpacked at /content/dataset."
        )

os.makedirs(SUPP_ROOT, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

# The val/test roots of the released archive can contain extra items that
# are not one of the study's 39 classes. Step 4 of notebook 01 counted only
# the 39 class names present in train, hence 634/636. Here the extra items
# are quarantined against the fixed class list before a strict recursive
# count is taken.
TARGET_CLASSES = {
    "badou", "baifuzi", "baiguo", "banxia", "beidougen", "caowu",
    "changshan", "chonglou", "chuanlianzi", "gansui", "heshi",
    "hongdaji", "huajiao", "jili", "jiulixiang", "kulianpi", "langdu",
    "liangmianzhen", "maqianzi", "mianmaguanzhong", "mubiezi",
    "naoyanghua", "qianjinzi", "qianniuzi", "shancigu", "shanglu",
    "shechuangzi", "tiannanxing", "tianxianzi", "tujingpi", "wuzhuyu",
    "xiangjiapi", "xiangsizi", "xianmao", "yadanzi", "yangjinhua",
    "yingsuqiao", "yuanhua", "zhuyazao",
}
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
QUARANTINE_ROOT = f"/content/dataset_excluded_non39_{int(time.time())}"
excluded_items = []

for split in ["train", "val", "test"]:
    split_root = os.path.join(DATA_ROOT, split)
    for item_name in sorted(os.listdir(split_root)):
        if item_name in TARGET_CLASSES:
            continue
        src = os.path.join(split_root, item_name)
        image_count = (
            sum(
                1
                for dp, _, files in os.walk(src)
                for fn in files
                if fn.lower().endswith(IMG_EXTS)
            )
            if os.path.isdir(src)
            else int(item_name.lower().endswith(IMG_EXTS))
        )
        dst_dir = os.path.join(QUARANTINE_ROOT, split)
        os.makedirs(dst_dir, exist_ok=True)
        dst = os.path.join(dst_dir, item_name)
        if os.path.exists(dst):
            dst = os.path.join(dst_dir, f"{int(time.time() * 1000)}_{item_name}")
        shutil.move(src, dst)
        excluded_items.append({
            "split": split,
            "item": item_name,
            "reason": "non-39-class item in the split root",
            "item_type": "directory" if os.path.isdir(dst) else "file",
            "image_count": image_count,
            "quarantine_path": dst,
        })

# Also clean nested directories and hidden image copies inside the 39 class
# folders. The expected layout is split/class/image, with no sub-directory
# below a class folder. Colab/Jupyter .ipynb_checkpoints directories copy
# images and would inflate a recursive count.
for split in ["train", "val", "test"]:
    for cls in sorted(TARGET_CLASSES):
        class_dir = os.path.join(DATA_ROOT, split, cls)
        if not os.path.isdir(class_dir):
            continue
        for item_name in sorted(os.listdir(class_dir)):
            src = os.path.join(class_dir, item_name)
            is_nested_dir = os.path.isdir(src)
            is_hidden_image = (
                os.path.isfile(src)
                and item_name.startswith(".")
                and item_name.lower().endswith(IMG_EXTS)
            )
            if not (is_nested_dir or is_hidden_image):
                continue
            image_count = (
                sum(
                    1
                    for dp, _, files in os.walk(src)
                    for fn in files
                    if fn.lower().endswith(IMG_EXTS)
                )
                if is_nested_dir
                else 1
            )
            dst_dir = os.path.join(QUARANTINE_ROOT, split, cls)
            os.makedirs(dst_dir, exist_ok=True)
            dst = os.path.join(dst_dir, item_name)
            if os.path.exists(dst):
                dst = os.path.join(dst_dir, f"{int(time.time() * 1000)}_{item_name}")
            shutil.move(src, dst)
            excluded_items.append({
                "split": split,
                "item": f"{cls}/{item_name}",
                "reason": (
                    "nested directory inside a class folder"
                    if is_nested_dir
                    else "hidden image copy inside a class folder"
                ),
                "item_type": "directory" if is_nested_dir else "file",
                "image_count": image_count,
                "quarantine_path": dst,
            })

if excluded_items:
    excluded_df = pd.DataFrame(excluded_items)
    audit_path = os.path.join(TABLE_DIR, "excluded_non39_items_audit.csv")
    excluded_df.to_csv(audit_path, index=False, encoding="utf-8-sig")
    print("Quarantined items outside the 39 classes "
          "(only the temporary /content extraction tree is moved):")
    print(excluded_df.to_string(index=False))
    print(f"Quarantine audit table written to: {audit_path}")
else:
    print("No items outside the 39 classes were found.")

# Same rule as step 6 of notebook 01: once the data directory changes the
# stale Ultralytics cache must be deleted, otherwise later scans keep
# reusing the pre-cleaning class and sample index.
removed_cache_files = []
for split in ["train", "val", "test", "test_subset"]:
    cache_path = os.path.join(DATA_ROOT, f"{split}.cache")
    if os.path.exists(cache_path):
        os.remove(cache_path)
        removed_cache_files.append(cache_path)
if removed_cache_files:
    print("Removed stale Ultralytics caches:")
    for cache_path in removed_cache_files:
        print(f"  - {cache_path}")
else:
    print("No stale Ultralytics data cache found.")

if not os.path.exists(PRETRAINED_WEIGHT):
    rc = os.system(
        f'wget -q "{PRETRAINED_URL}" -O "{PRETRAINED_WEIGHT}"'
    )
    if rc != 0:
        raise RuntimeError("Failed to download yolov12s-cls.pt; check the network.")

if os.path.getsize(PRETRAINED_WEIGHT) < 1_000_000:
    raise RuntimeError("yolov12s-cls.pt is incomplete.")

def md5sum(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

class_names = sorted(TARGET_CLASSES)

for split in ["train", "val", "test"]:
    split_root = os.path.join(DATA_ROOT, split)
    actual_classes = {
        d for d in os.listdir(split_root)
        if os.path.isdir(os.path.join(split_root, d))
    }
    if actual_classes != TARGET_CLASSES:
        missing = sorted(TARGET_CLASSES - actual_classes)
        extra = sorted(actual_classes - TARGET_CLASSES)
        raise RuntimeError(
            f"{split} class directories do not match. missing={missing}, "
            f"extra={extra}. Do not continue training."
        )

dataset_counts = {}
for split in ["train", "val", "test"]:
    root = os.path.join(DATA_ROOT, split)
    dataset_counts[split] = sum(
        1
        for cls in TARGET_CLASSES
        for dp, _, files in os.walk(os.path.join(root, cls))
        for fn in files
        if fn.lower().endswith(IMG_EXTS)
    )

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "ultralytics": __import__("ultralytics").__version__,
    "pretrained_weight": PRETRAINED_WEIGHT,
    "pretrained_weight_md5": md5sum(PRETRAINED_WEIGHT),
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "seeds": SEEDS,
    "variants": VARIANTS,
    "class_count": len(class_names),
    "dataset_counts": dataset_counts,
    "excluded_non39_items": excluded_items,
}

with open(os.path.join(SUPP_ROOT, "environment_and_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(environment, f, ensure_ascii=False, indent=2)

print("=" * 72)
print("Environment and dataset verification")
print("=" * 72)
print(json.dumps(environment, ensure_ascii=False, indent=2))

if len(class_names) != 39:
    raise RuntimeError(f"Expected 39 classes, found {len(class_names)}.")
if dataset_counts != {"train": 1981, "val": 634, "test": 636}:
    raise RuntimeError(
        "Split sizes do not match the study. Expected "
        "train/val/test = 1981/634/636, found "
        f"{dataset_counts}. Do not continue training."
    )
print("\nEnvironment check passed: exactly 39 classes, "
      "1981/634/636. Proceed to step 2.")


In [ ]:
# ============================================================
# Step 2: define SE, standard CBAM, the original LTA, and one shared
# injection path used identically by all variants
#
# Design constraints:
# - all three modules are inserted at the same layer, len(backbone)-2
# - SE, CBAM and LTA all use reduction=16
# - CBAM uses standard channel attention plus 7x7 spatial attention
# - LTA keeps the original experiment's zero-initialization exactly as it
#   was; earlier results are not retroactively changed
# ============================================================

import torch
import torch.nn as nn


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(self.pool(x))


class CBAMChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class CBAMSpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(
            2, 1, kernel_size, padding=kernel_size // 2, bias=False
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = CBAMChannelAttention(channels, reduction)
        self.spatial_att = CBAMSpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        return x * self.spatial_att(x)


class LTAChannelAttention(nn.Module):
    """Identical to steps 10-11 of notebook 01: the final 1x1 convolution
    is zero-initialized."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.fc[2].weight)

    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))


class LTASpatialAttention(nn.Module):
    """Identical to steps 10-11 of notebook 01: the 7x7 spatial convolution
    is zero-initialized."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(
            2, 1, kernel_size, padding=kernel_size // 2, bias=False
        )
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.conv.weight)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class LTABlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = LTAChannelAttention(channels, reduction)
        self.spatial_att = LTASpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        return x * self.spatial_att(x)


class AttentionWrapper(nn.Module):
    """Top-level nn.Module so that PyTorch checkpointing serializes it."""
    def __init__(self, orig_layer, attention):
        super().__init__()
        self.orig_layer = orig_layer
        self.attention = attention
        self.f = getattr(orig_layer, "f", -1)
        self.i = getattr(orig_layer, "i", None)
        self.type = getattr(orig_layer, "type", None)

    def forward(self, x):
        return self.attention(self.orig_layer(x))


def build_attention(kind, channels):
    if kind == "SE":
        return SEBlock(channels)
    if kind == "CBAM":
        return CBAMBlock(channels)
    if kind == "LTA":
        return LTABlock(channels)
    raise ValueError(f"Unknown attention module: {kind}")


def probe_output_channels(real_model, layer_idx, device, imgsz=224):
    target_layer = real_model.model[layer_idx]
    captured = {}

    def hook(_, __, output):
        captured["channels"] = int(output.shape[1])

    handle = target_layer.register_forward_hook(hook)
    was_training = real_model.training
    real_model.eval()
    with torch.no_grad():
        real_model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    real_model.train(was_training)
    handle.remove()

    if "channels" not in captured:
        raise RuntimeError("Could not probe the target layer output channels.")
    return captured["channels"]


def attach_attention(real_model, kind, layer_idx, channels, device):
    original_layer = real_model.model[layer_idx]
    attention = build_attention(kind, channels).to(device)
    real_model.model[layer_idx] = AttentionWrapper(
        original_layer, attention
    ).to(device)
    return attention


def make_injection_callback(kind, status):
    def callback(trainer):
        if status["injected"]:
            return

        real_model = trainer.model
        device = next(real_model.parameters()).device
        layer_idx = len(real_model.model) - 2
        channels = probe_output_channels(
            real_model, layer_idx, device, imgsz=IMG_SIZE
        )

        attention = attach_attention(
            real_model, kind, layer_idx, channels, device
        )
        status.update({
            "injected": True,
            "layer_idx": layer_idx,
            "channels": channels,
            "module_params": sum(p.numel() for p in attention.parameters()),
        })

        if trainer.optimizer is None:
            raise RuntimeError("trainer.optimizer is missing; the inserted module "
                               "would not be trained.")

        param_ids_before = {
            id(p)
            for group in trainer.optimizer.param_groups
            for p in group["params"]
        }
        new_params = [
            p for p in attention.parameters()
            if id(p) not in param_ids_before
        ]
        if not new_params:
            raise RuntimeError("No attention parameters found to register with "
                               "the optimizer.")

        decay_group = next(
            (
                g for g in trainer.optimizer.param_groups
                if float(g.get("weight_decay", 0)) > 0
            ),
            trainer.optimizer.param_groups[0],
        )
        decay_group["params"].extend(new_params)
        status["optimizer_synced"] = True

        if not (
            hasattr(trainer, "ema")
            and trainer.ema is not None
            and hasattr(trainer.ema, "ema")
        ):
            raise RuntimeError("No EMA model found; best.pt structure cannot "
                               "be guaranteed.")

        attach_attention(
            trainer.ema.ema, kind, layer_idx, channels, device
        )
        status["ema_synced"] = True

        total_params = sum(p.numel() for p in real_model.parameters())
        status["total_params"] = total_params
        print(
            f"[{kind}] layer={layer_idx}, channels={channels}, "
            f"module params={status['module_params']:,}, "
            f"total params={total_params:,}"
        )

    return callback


print("SE / CBAM / LTA and the shared injection path are defined.")
print("Note: LTA is structurally identical to standard CBAM. This "
      "experiment therefore tests whether the original LTA's")
print("zero-initialization strategy yields a reproducible advantage. "
      "The manuscript states this explicitly.")


In [ ]:
# ============================================================
# Step 3: run 4 variants x 3 seeds
#
# This is the long step. Each finished run is written to Drive immediately.
# If Colab disconnects, rerun steps 1-3:
# - runs that already have a verified best.pt are skipped
# - unfinished runs are trained again from scratch
# ============================================================

import gc
import os
import json
import random
import shutil
import time
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO


def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Independent hard gate: even if step 1 is skipped, this step must never
# start on the wrong data.
expected_counts = {"train": 1981, "val": 634, "test": 636}
step22_counts = {}
step22_class_sets = {}
for split in ["train", "val", "test"]:
    split_root = os.path.join(DATA_ROOT, split)
    if not os.path.isdir(split_root):
        raise RuntimeError(f"Aborted: {split_root} not found. Run step 1 first.")
    step22_class_sets[split] = {
        d for d in os.listdir(split_root)
        if os.path.isdir(os.path.join(split_root, d))
    }
    step22_counts[split] = sum(
        1
        for cls in step22_class_sets[split]
        for dp, _, files in os.walk(os.path.join(split_root, cls))
        for fn in files
        if fn.lower().endswith(IMG_EXTS)
    )

bad_class_sets = {
    split: {
        "missing": sorted(TARGET_CLASSES - classes),
        "extra": sorted(classes - TARGET_CLASSES),
    }
    for split, classes in step22_class_sets.items()
    if classes != TARGET_CLASSES
}
if bad_class_sets or step22_counts != expected_counts:
    raise RuntimeError(
        "Hard stop: training on incorrect data is not allowed. "
        f"class mismatch={bad_class_sets}; counts={step22_counts}; "
        f"expected={expected_counts}. Rerun step 1."
    )
print("Second data gate passed: 39 classes, "
      "train/val/test = 1981/634/636.")


task_rows = []

for variant in VARIANTS:
    for seed in SEEDS:
        run_name = f"v12s_{variant.lower()}_seed{seed}"
        run_dir = os.path.join(SUPP_ROOT, run_name)
        best_path = os.path.join(run_dir, "weights", "best.pt")
        status_path = os.path.join(run_dir, "supplement_status.json")

        print("\n" + "=" * 78)
        print(f"Run: {variant} | seed={seed}")
        print("=" * 78)

        completed_status = {}
        if os.path.exists(status_path):
            try:
                with open(status_path, "r", encoding="utf-8") as f:
                    completed_status = json.load(f)
            except Exception:
                completed_status = {}

        is_verified_complete = (
            os.path.exists(best_path)
            and completed_status.get("status") == "completed"
            and completed_status.get("variant") == variant
            and completed_status.get("seed") == seed
        )
        if is_verified_complete:
            print(f"Verified completion marker present, skipping: {best_path}")
            task_rows.append({
                "variant": variant,
                "seed": seed,
                "status": "verified_completed_checkpoint",
                "best_path": best_path,
            })
            continue

        if os.path.isdir(run_dir):
            preserved_dir = (
                run_dir + f"_INVALID_OR_INCOMPLETE_{int(time.time())}"
            )
            shutil.move(run_dir, preserved_dir)
            print(f"Run without a completion marker found; preserved outside the "
                  f"analysis directory: {preserved_dir}")

        set_all_seeds(seed)
        model = YOLO(PRETRAINED_WEIGHT)
        injection_status = {
            "variant": variant,
            "seed": seed,
            "injected": variant == "baseline",
            "optimizer_synced": variant == "baseline",
            "ema_synced": variant == "baseline",
        }

        if variant != "baseline":
            model.add_callback(
                "on_pretrain_routine_end",
                make_injection_callback(variant, injection_status),
            )

        try:
            model.train(
                data=DATA_ROOT,
                epochs=EPOCHS,
                imgsz=IMG_SIZE,
                batch=BATCH_SIZE,
                patience=PATIENCE,
                seed=seed,
                deterministic=True,
                project=SUPP_ROOT,
                name=run_name,
                exist_ok=True,
                plots=True,
                verbose=True,
            )

            if not os.path.exists(best_path):
                raise RuntimeError(f"Training finished but {best_path} is missing")

            if variant != "baseline" and not all(
                injection_status.get(k, False)
                for k in ["injected", "optimizer_synced", "ema_synced"]
            ):
                raise RuntimeError(
                    f"{variant} injection self-check failed: {injection_status}"
                )

            injection_status["status"] = "completed"
            injection_status["best_path"] = best_path
            injection_status["dataset_counts"] = step22_counts
            injection_status["pretrained_weight_md5"] = md5sum(PRETRAINED_WEIGHT)
            with open(status_path, "w", encoding="utf-8") as f:
                json.dump(injection_status, f, ensure_ascii=False, indent=2)

            task_rows.append({
                "variant": variant,
                "seed": seed,
                "status": "completed",
                "best_path": best_path,
            })
            print(f"Completed and saved: {best_path}")

        except Exception as e:
            injection_status["status"] = "failed"
            injection_status["error"] = repr(e)
            os.makedirs(run_dir, exist_ok=True)
            with open(status_path, "w", encoding="utf-8") as f:
                json.dump(injection_status, f, ensure_ascii=False, indent=2)
            task_rows.append({
                "variant": variant,
                "seed": seed,
                "status": "failed",
                "best_path": None,
                "error": repr(e),
            })
            pd.DataFrame(task_rows).to_csv(
                os.path.join(TABLE_DIR, "training_task_status.csv"),
                index=False,
            )
            raise
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        pd.DataFrame(task_rows).to_csv(
            os.path.join(TABLE_DIR, "training_task_status.csv"),
            index=False,
        )

task_df = pd.DataFrame(task_rows)
print("\n" + "=" * 78)
print("Status of all training runs")
print("=" * 78)
print(task_df.to_string(index=False))

missing = []
for variant in VARIANTS:
    for seed in SEEDS:
        run_dir = os.path.join(
            SUPP_ROOT,
            f"v12s_{variant.lower()}_seed{seed}",
        )
        p = os.path.join(run_dir, "weights", "best.pt")
        status_p = os.path.join(run_dir, "supplement_status.json")
        completed = {}
        if os.path.exists(status_p):
            try:
                with open(status_p, "r", encoding="utf-8") as f:
                    completed = json.load(f)
            except Exception:
                completed = {}
        if not (os.path.exists(p) and completed.get("status") == "completed"):
            missing.append(run_dir)

if missing:
    print("\nThe following runs are not finished; rerun this step:")
    print("\n".join(missing))
else:
    print("\nAll 12 checkpoints are complete. Proceed to step 4.")


In [ ]:
# ============================================================
# Step 4: per-image evaluation of all YOLO runs
#
# Outputs:
# - one predictions CSV per variant/seed (636 rows)
# - attention_ablation_metrics.csv
# - Top-1, Top-5, parameter count, GFLOPs and mean inference time
# ============================================================

import gc
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_paths = sorted(
    str(p)
    for p in Path(TEST_DIR).glob("*/*")
    if p.suffix.lower() in IMG_EXTS
)
if len(test_paths) != 636:
    raise RuntimeError(f"Expected 636 test images, found {len(test_paths)}.")


def names_to_dict(names):
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {i: str(v) for i, v in enumerate(names)}


metric_rows = []

for variant in VARIANTS:
    for seed in SEEDS:
        run_name = f"v12s_{variant.lower()}_seed{seed}"
        best_path = os.path.join(
            SUPP_ROOT, run_name, "weights", "best.pt"
        )
        pred_path = os.path.join(
            PRED_DIR, f"{run_name}_test_predictions.csv"
        )

        print("\n" + "=" * 78)
        print(f"Per-image evaluation: {variant} | seed={seed}")
        print("=" * 78)

        if not os.path.exists(best_path):
            raise FileNotFoundError(
                f"{best_path} is missing; complete step 3 first."
            )

        model = YOLO(best_path)
        names = names_to_dict(model.names)
        name_to_idx = {v: k for k, v in names.items()}

        unknown_classes = sorted(
            {Path(p).parent.name for p in test_paths} - set(name_to_idx)
        )
        if unknown_classes:
            raise RuntimeError(
                f"Checkpoint class names do not match the test directory: "
                f"{unknown_classes}"
            )

        n_params = sum(p.numel() for p in model.model.parameters())
        try:
            from ultralytics.utils.torch_utils import get_flops
            gflops = float(get_flops(model.model, imgsz=IMG_SIZE))
        except Exception as e:
            print(f"GFLOPs computation failed, recorded as empty: {e}")
            gflops = np.nan

        rows = []
        inference_ms = []
        stream = model.predict(
            source=test_paths,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            device=0 if torch.cuda.is_available() else "cpu",
            stream=True,
            verbose=False,
        )

        for expected_path, result in zip(test_paths, stream):
            true_name = Path(expected_path).parent.name
            true_idx = int(name_to_idx[true_name])
            top1_idx = int(result.probs.top1)
            top5_idx = [int(x) for x in result.probs.top5]
            conf = result.probs.data.detach().cpu().numpy()
            inference_ms.append(float(result.speed.get("inference", np.nan)))
            rows.append({
                "relative_path": str(
                    Path(expected_path).relative_to(TEST_DIR)
                ),
                "true_idx": true_idx,
                "true_class": true_name,
                "pred_idx": top1_idx,
                "pred_class": names[top1_idx],
                "top1_confidence": float(conf[top1_idx]),
                "top1_correct": int(top1_idx == true_idx),
                "top5_indices": "|".join(map(str, top5_idx)),
                "top5_correct": int(true_idx in top5_idx),
            })

        pred_df = pd.DataFrame(rows)
        if len(pred_df) != 636:
            raise RuntimeError(
                f"{run_name} produced only {len(pred_df)} predictions; 636 expected."
            )
        if pred_df["relative_path"].duplicated().any():
            raise RuntimeError(f"{run_name} contains duplicate image paths.")

        pred_df.to_csv(pred_path, index=False)
        top1 = 100.0 * pred_df["top1_correct"].mean()
        top5 = 100.0 * pred_df["top5_correct"].mean()
        mean_ms = float(np.nanmean(inference_ms))

        metric_rows.append({
            "variant": variant,
            "seed": seed,
            "top1_pct": top1,
            "top5_pct": top5,
            "params": n_params,
            "gflops": gflops,
            "mean_inference_ms_per_image": mean_ms,
            "n_test": len(pred_df),
            "checkpoint": best_path,
            "prediction_csv": pred_path,
        })
        pd.DataFrame(metric_rows).to_csv(
            os.path.join(TABLE_DIR, "attention_ablation_metrics.csv"),
            index=False,
        )
        print(
            f"Top-1={top1:.2f}% | Top-5={top5:.2f}% | "
            f"Params={n_params:,} | GFLOPs={gflops:.3f} | "
            f"Inference={mean_ms:.3f} ms/image"
        )

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

metrics_df = pd.DataFrame(metric_rows)
print("\nPer-image evaluation of the YOLO runs is complete")
print(metrics_df.to_string(index=False))


In [ ]:
# ============================================================
# Step 5: complete the EfficientNet-B0 Top-5 and per-image predictions
#
# EfficientNet is not retrained here; this reads the best.pt produced by
# step 18 of notebook 01.
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

EFF_WEIGHTS = (
    "/content/drive/MyDrive/TCM_YOLOv12_runs/"
    "efficientnet_b0/best.pt"
)
EFF_PRED_CSV = os.path.join(
    PRED_DIR, "efficientnet_b0_test_predictions.csv"
)

if not os.path.exists(EFF_WEIGHTS):
    raise FileNotFoundError(
        f"{EFF_WEIGHTS} not found. Run step 18 of notebook 01 to train "
        f"EfficientNet-B0 first."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225],
    ),
])
test_ds = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

eff_model = models.efficientnet_b0(weights=None)
in_features = eff_model.classifier[1].in_features
eff_model.classifier[1] = nn.Linear(in_features, len(test_ds.classes))
eff_model.load_state_dict(torch.load(EFF_WEIGHTS, map_location=device))
eff_model = eff_model.to(device)
eff_model.eval()

rows = []
sample_offset = 0
elapsed_ms = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = eff_model(images)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed_ms.extend([
            (time.perf_counter() - t0) * 1000.0 / images.size(0)
        ] * images.size(0))

        probs = torch.softmax(logits, dim=1)
        top5_prob, top5_idx = probs.topk(5, dim=1)
        top1_idx = top5_idx[:, 0]

        for j in range(images.size(0)):
            dataset_path, dataset_true = test_ds.samples[sample_offset + j]
            true_idx = int(labels[j].item())
            if true_idx != int(dataset_true):
                raise RuntimeError("ImageFolder sample order disagrees with the "
                                   "DataLoader labels.")
            pred_idx = int(top1_idx[j].item())
            indices = [int(x) for x in top5_idx[j].cpu().tolist()]
            rows.append({
                "relative_path": str(
                    Path(dataset_path).relative_to(TEST_DIR)
                ),
                "true_idx": true_idx,
                "true_class": test_ds.classes[true_idx],
                "pred_idx": pred_idx,
                "pred_class": test_ds.classes[pred_idx],
                "top1_confidence": float(top5_prob[j, 0].item()),
                "top1_correct": int(pred_idx == true_idx),
                "top5_indices": "|".join(map(str, indices)),
                "top5_correct": int(true_idx in indices),
            })
        sample_offset += images.size(0)

eff_pred_df = pd.DataFrame(rows)
if len(eff_pred_df) != 636:
    raise RuntimeError(
        f"EfficientNet produced {len(eff_pred_df)} predictions; 636 expected."
    )
eff_pred_df.to_csv(EFF_PRED_CSV, index=False)

eff_top1 = 100.0 * eff_pred_df["top1_correct"].mean()
eff_top5 = 100.0 * eff_pred_df["top5_correct"].mean()
eff_params = sum(p.numel() for p in eff_model.parameters())

eff_row = pd.DataFrame([{
    "variant": "EfficientNet-B0",
    "seed": 42,
    "top1_pct": eff_top1,
    "top5_pct": eff_top5,
    "params": eff_params,
    "gflops": np.nan,
    "mean_inference_ms_per_image": float(np.mean(elapsed_ms)),
    "n_test": len(eff_pred_df),
    "checkpoint": EFF_WEIGHTS,
    "prediction_csv": EFF_PRED_CSV,
}])
eff_row.to_csv(
    os.path.join(TABLE_DIR, "efficientnet_b0_metrics_complete.csv"),
    index=False,
)

print("=" * 72)
print("EfficientNet-B0 test-set results")
print("=" * 72)
print(f"Top-1: {eff_top1:.2f}%")
print(f"Top-5: {eff_top5:.2f}%")
print(f"Parameters: {eff_params:,}")
print(f"Per-image predictions: {EFF_PRED_CSV}")


In [ ]:
# ============================================================
# Step 6: paired bootstrap 95% CIs and multi-seed summaries
#
# Statistical rules:
# - single-model accuracy CI: resample the 636 test images with replacement
# - difference CI: apply the same resampled image indices to both models
#   within one replicate (paired bootstrap)
# - the test set is never used for tuning; this step only performs
#   inference on the final checkpoints
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd

N_BOOT = 10000
BOOTSTRAP_SEED = 20260827
rng = np.random.default_rng(BOOTSTRAP_SEED)


def load_correctness(csv_path):
    df = pd.read_csv(csv_path).sort_values("relative_path").reset_index(drop=True)
    if len(df) != 636:
        raise RuntimeError(f"{csv_path} does not have 636 rows.")
    if df["relative_path"].duplicated().any():
        raise RuntimeError(f"{csv_path} contains duplicate paths.")
    return df["relative_path"], df["top1_correct"].to_numpy(dtype=np.int8)


def bootstrap_accuracy(correct, n_boot=N_BOOT):
    n = len(correct)
    values = np.empty(n_boot, dtype=np.float64)
    for start in range(0, n_boot, 500):
        stop = min(start + 500, n_boot)
        idx = rng.integers(0, n, size=(stop - start, n))
        values[start:stop] = correct[idx].mean(axis=1) * 100.0
    return (
        float(correct.mean() * 100.0),
        float(np.percentile(values, 2.5)),
        float(np.percentile(values, 97.5)),
    )


def paired_bootstrap(a, b, n_boot=N_BOOT):
    """Return the point estimate of a-b, its 95% CI, and the two-sided
    bootstrap p-value."""
    if len(a) != len(b):
        raise ValueError("Paired bootstrap requires equal sample counts.")
    diff = a.astype(np.float64) - b.astype(np.float64)
    n = len(diff)
    values = np.empty(n_boot, dtype=np.float64)
    for start in range(0, n_boot, 500):
        stop = min(start + 500, n_boot)
        idx = rng.integers(0, n, size=(stop - start, n))
        values[start:stop] = diff[idx].mean(axis=1) * 100.0
    point = float(diff.mean() * 100.0)
    lo, hi = np.percentile(values, [2.5, 97.5])
    p_two_sided = float(
        min(
            1.0,
            2.0 * min(
                np.mean(values <= 0.0),
                np.mean(values >= 0.0),
            ),
        )
    )
    return point, float(lo), float(hi), p_two_sided


metric_path = os.path.join(TABLE_DIR, "attention_ablation_metrics.csv")
metrics_df = pd.read_csv(metric_path)

ci_rows = []
pair_rows = []

for _, row in metrics_df.iterrows():
    paths, correct = load_correctness(row["prediction_csv"])
    acc, lo, hi = bootstrap_accuracy(correct)
    ci_rows.append({
        "variant": row["variant"],
        "seed": int(row["seed"]),
        "top1_pct": acc,
        "top1_ci95_low": lo,
        "top1_ci95_high": hi,
        "n_test": len(correct),
    })

for seed in SEEDS:
    base_row = metrics_df[
        (metrics_df["variant"] == "baseline")
        & (metrics_df["seed"] == seed)
    ].iloc[0]
    base_paths, base_correct = load_correctness(
        base_row["prediction_csv"]
    )

    for variant in ["SE", "CBAM", "LTA"]:
        comp_row = metrics_df[
            (metrics_df["variant"] == variant)
            & (metrics_df["seed"] == seed)
        ].iloc[0]
        comp_paths, comp_correct = load_correctness(
            comp_row["prediction_csv"]
        )
        if not base_paths.equals(comp_paths):
            raise RuntimeError(
                f"Image order differs between baseline and {variant} at seed={seed}."
            )

        delta, lo, hi, p_boot = paired_bootstrap(
            comp_correct, base_correct
        )
        pair_rows.append({
            "comparison": f"{variant} - baseline",
            "seed": seed,
            "delta_top1_pp": delta,
            "delta_ci95_low_pp": lo,
            "delta_ci95_high_pp": hi,
            "bootstrap_p_two_sided": p_boot,
            "ci_excludes_zero": bool(lo > 0 or hi < 0),
        })

ci_df = pd.DataFrame(ci_rows)
pair_df = pd.DataFrame(pair_rows)

seed_summary = (
    metrics_df.groupby("variant", as_index=False)
    .agg(
        top1_mean_pct=("top1_pct", "mean"),
        top1_sd_pct=("top1_pct", "std"),
        top5_mean_pct=("top5_pct", "mean"),
        top5_sd_pct=("top5_pct", "std"),
        params=("params", "first"),
        gflops=("gflops", "first"),
        inference_ms_mean=("mean_inference_ms_per_image", "mean"),
    )
)

delta_seed_summary = (
    pair_df.groupby("comparison", as_index=False)
    .agg(
        delta_mean_pp=("delta_top1_pp", "mean"),
        delta_sd_pp=("delta_top1_pp", "std"),
        seeds=("seed", "count"),
    )
)

ci_df.to_csv(
    os.path.join(TABLE_DIR, "bootstrap_model_accuracy_ci.csv"),
    index=False,
)
pair_df.to_csv(
    os.path.join(TABLE_DIR, "bootstrap_paired_differences.csv"),
    index=False,
)
seed_summary.to_csv(
    os.path.join(TABLE_DIR, "multiseed_summary.csv"),
    index=False,
)
delta_seed_summary.to_csv(
    os.path.join(TABLE_DIR, "multiseed_delta_summary.csv"),
    index=False,
)

print("\n" + "=" * 80)
print("Multi-seed summary")
print("=" * 80)
print(seed_summary.to_string(index=False))

print("\n" + "=" * 80)
print("Per-seed paired bootstrap: attention variant minus baseline")
print("=" * 80)
print(pair_df.to_string(index=False))

print("\n" + "=" * 80)
print("Across-seed summary of the differences")
print("=" * 80)
print(delta_seed_summary.to_string(index=False))


In [ ]:
# ============================================================
# Step 7: completeness check and release packaging
#
# Verifies that every required artefact exists, records size + MD5 for
# each one in result_manifest.json, and packages the release archive.
# Fails loudly if any run, prediction file or table is missing.
# ============================================================

import json
import os
import shutil
from pathlib import Path

required_files = [
    os.path.join(SUPP_ROOT, "environment_and_protocol.json"),
    os.path.join(TABLE_DIR, "attention_ablation_metrics.csv"),
    os.path.join(TABLE_DIR, "efficientnet_b0_metrics_complete.csv"),
    os.path.join(TABLE_DIR, "bootstrap_model_accuracy_ci.csv"),
    os.path.join(TABLE_DIR, "bootstrap_paired_differences.csv"),
    os.path.join(TABLE_DIR, "multiseed_summary.csv"),
    os.path.join(TABLE_DIR, "multiseed_delta_summary.csv"),
]

for variant in VARIANTS:
    for seed in SEEDS:
        required_files.append(
            os.path.join(
                PRED_DIR,
                f"v12s_{variant.lower()}_seed{seed}_test_predictions.csv",
            )
        )
required_files.append(
    os.path.join(PRED_DIR, "efficientnet_b0_test_predictions.csv")
)

missing = [p for p in required_files if not os.path.exists(p)]
if missing:
    print("Incomplete results; the following files are missing:")
    print("\n".join(missing))
    raise RuntimeError("Complete the missing steps before packaging.")

manifest = []
for p in required_files:
    manifest.append({
        "path": p,
        "size_bytes": os.path.getsize(p),
        "md5": md5sum(p),
    })

manifest_path = os.path.join(SUPP_ROOT, "result_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

zip_base = (
    "/content/drive/MyDrive/"
    "TCM_supplementary_experiment_results"
)
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=SUPP_ROOT,
)

print("=" * 80)
print("All artefacts are present")
print("=" * 80)
print(f"Results directory: {SUPP_ROOT}")
print(f"Archive: {zip_path}")
print("\nThis archive is the released artefact set for the study.")
